# 05 — Sentence Embeddings & Semantic Search

Where TF-IDF measures **shared words**, embeddings measure **shared meaning**.
This notebook: encode → cosine → rank, then wire an embedding-based intent
path next to the TF-IDF one.

Production use: `app/services/embedding_service.py`,
`similarity_service.py`, `EmbeddingIntentClassifier` ·
Docs: `docs/nlp/05_embeddings.md`

In [ ]:
import sys, time
from pathlib import Path

cwd = Path.cwd()
backend_dir = cwd.parent / "backend" if cwd.name == "notebooks" else cwd / "backend"
sys.path.insert(0, str(backend_dir))

import numpy as np
from app.services.embedding_service import get_embedding_service
from app.services.similarity_service import SimilarityService, cosine

t0 = time.time()
emb = get_embedding_service()
print(f"model loaded: {emb.model_name}, dim={emb.dim} ({time.time()-t0:.1f}s)")

## 1. Encoding — text to unit vectors

Each sentence becomes a `dim`-length vector, L2-normalized so **cosine
similarity = dot product**. Watch the numbers: paraphrases score ~0.5–0.8,
unrelated text ~0.0.

In [ ]:
texts = [
    "I forgot my password",
    "I can't remember my login password",
    "what is the weather like",
    "the stock market rallied today",
]
vectors = emb.encode(texts)
print("shape:", vectors.shape, "| row norms:", np.round(np.linalg.norm(vectors, axis=1), 3))

def sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"paraphrase pair : {sim(vectors[0], vectors[1]):.3f}")
print(f"pwd vs weather  : {sim(vectors[0], vectors[2]):.3f}")
print(f"pwd vs stocks   : {sim(vectors[0], vectors[3]):.3f}")

## 2. The failure TF-IDF cannot fix

TF-IDF cosine needs shared *strings*. These two sentences mean the same
thing but share almost no vocabulary — TF-IDF says "unrelated", embeddings
say "paraphrase". That gap is the reason Phase 5 exists.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pair = ["I forgot my password", "I can't remember my login password"]
tf = TfidfVectorizer().fit_transform(pair)
tfidf_score = float(cosine_similarity(tf[0], tf[1])[0][0])
emb_score = cosine(emb.embed(pair[0]), emb.embed(pair[1]))
print(f"same meaning, different words:")
print(f"  TF-IDF cosine : {tfidf_score:.3f}   <- near zero: 'unrelated'")
print(f"  embedding     : {emb_score:.3f}   <- clearly related")

## 3. Semantic ranking — the retrieval primitive

`SimilarityService.rank(query, candidates)` is what Phase 7's RAG will call:
question in → document chunks ordered by meaning.

In [ ]:
svc = SimilarityService(emb)
candidates = [
    "i cannot remember my login password",
    "reset my login credentials",
    "the stock market rose today",
    "what is the weather like",
]
for m in svc.rank("I forgot my password", candidates):
    print(f"  {m.score:.3f}  {m.text}")

## 4. Thresholds — when to say "I don't know"

Cosine scores cluster by relationship type (see docs). Below
`EMBEDDING_UNKNOWN_THRESHOLD = 0.45` the intent path returns `unknown`.

In [ ]:
from app.nlp.intent import EMBEDDING_UNKNOWN_THRESHOLD

query = emb.embed("zxcvbn mnbvc qwe rty")
top = max(cosine(query, emb.embed(t)) for t in candidates)
print(f"best score vs dataset-like text: {top:.3f}")
print("-> unknown" if top < EMBEDDING_UNKNOWN_THRESHOLD else "-> accepted")

## 5. Embedding intent backend

`EmbeddingIntentClassifier` embeds all 489 patterns once, then classifies
each query by nearest pattern in embedding space — same aggregation rule as
the TF-IDF baseline, different geometry.

Opt-in via `classify_intent(text, backend="embedding")` (default stays
sklearn: microseconds vs milliseconds).

In [ ]:
from app.nlp.intent import classify_intent

cases = [
    "hey, what's up?",
    "i cannot access my account i forgot the password",  # paraphrase, few shared keywords
    "what time is it",
    "zxcvbn mnbvcxz qwe rty 9876",
]
for msg in cases:
    emb_res = classify_intent(msg, backend="embedding")
    tfidf_res = classify_intent(msg, backend="auto")
    print(f"{msg!r:58}")
    print(f"   embedding: {emb_res.label:16} {emb_res.confidence:.2f}   auto: {tfidf_res.label:16} {tfidf_res.confidence:.2f}")

## 6. Where each backend wins

- **Embeddings:** paraphrases, zero-keyword overlap, natural rephrasing.
- **TF-IDF/sklearn:** rare tokens, exact phrasings, 100× cheaper inference.
- **Both:** unknown/gibberish -> `unknown` below threshold.

Production default: sklearn-first chain; embedding path available for
semantic workloads and for RAG's query-side encoding in Phase 7.

## 7. Production behavior

- Model loads lazily once per process (`get_embedding_service`).
- Fresh clone without the HF cache downloads ~80 MB on first encode; if
  that fails, callers degrade (intent → TF-IDF chain) — the API never 500s.
- All vectors unit-length: downstream cosine is a dot product.